In [1]:
# ==========================================
# RESUME SCREENING (HR TECH) USING HF TRANSFORMERS
# ==========================================

import torch
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F

# ==========================================
# 1. LOAD PRETRAINED MODEL
# ==========================================
model_name = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# ==========================================
# 2. MEAN POOLING FUNCTION
# ==========================================
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]  # last hidden states
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

# ==========================================
# 3. ENCODING FUNCTION
# ==========================================
def encode(text):
    encoded_input = tokenizer(text, padding=True, truncation=True, return_tensors='pt')

    with torch.no_grad():
        model_output = model(**encoded_input)

    sentence_embedding = mean_pooling(model_output, encoded_input['attention_mask'])
    return sentence_embedding

# ==========================================
# 4. SIMILARITY FUNCTION
# ==========================================
def cosine_similarity(a, b):
    return F.cosine_similarity(a, b)

# ==========================================
# 5. SAMPLE DATA (RESUMES + JOB DESCRIPTION)
# ==========================================

job_description = """
We are looking for a Machine Learning Engineer with experience in Python,
deep learning, NLP, and Transformers. Knowledge of PyTorch is required.
"""

resumes = [
    """
    Experienced software engineer skilled in Python, Java, and web development.
    """,

    """
    Machine learning engineer with experience in Python, PyTorch, NLP,
    and transformer models. Worked on deep learning projects.
    """,

    """
    Data analyst with strong knowledge of SQL, Excel, and data visualization tools.
    """
]

# ==========================================
# 6. ENCODE JOB DESCRIPTION
# ==========================================
job_emb = encode(job_description)

# ==========================================
# 7. SCORE EACH RESUME
# ==========================================
scores = []

for i, resume in enumerate(resumes):
    resume_emb = encode(resume)
    score = cosine_similarity(job_emb, resume_emb).item()
    scores.append((i, score))

# ==========================================
# 8. RANK RESUMES
# ==========================================
scores = sorted(scores, key=lambda x: x[1], reverse=True)

# ==========================================
# 9. OUTPUT RESULTS
# ==========================================
print("\n=== RESUME RANKING ===\n")

for rank, (idx, score) in enumerate(scores, start=1):
    print(f"Rank {rank}")
    print(f"Resume {idx+1}")
    print(f"Match Score: {score:.4f}\n")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== RESUME RANKING ===

Rank 1
Resume 2
Match Score: 0.8633

Rank 2
Resume 1
Match Score: 0.5675

Rank 3
Resume 3
Match Score: 0.2703



In [6]:
import torch
import numpy as np
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

# ==========================================
# SAMPLE DATA
# ==========================================
job_description = """
We are looking for a Machine Learning Engineer with experience in Python,
deep learning, NLP, and Transformers.
"""

resumes = [
    "Software engineer with Java and web development experience",
    "ML engineer with Python, NLP, and deep learning experience",
    "Data analyst skilled in Excel and SQL"
]

# ==========================================
# =========================
# MODEL 1: BEFORE FINE-TUNING (MiniLM)
# =========================
# ==========================================

model_name = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer1 = AutoTokenizer.from_pretrained(model_name)
model1 = AutoModel.from_pretrained(model_name)

def mean_pooling(output, mask):
    token_embeddings = output[0]
    mask = mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)

def encode1(text):
    inputs = tokenizer1(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        out = model1(**inputs)
    return mean_pooling(out, inputs["attention_mask"])

def cosine(a, b):
    return F.cosine_similarity(a, b)

job_emb = encode1(job_description)

print("\n================ BEFORE FINE-TUNING (MiniLM) ================\n")

for i, r in enumerate(resumes):
    emb = encode1(r)
    score = cosine(job_emb, emb).item()
    print(f"Resume {i+1} Score: {score:.4f}")

# ==========================================
# =========================
# MODEL 2: AFTER FINE-TUNING (DistilBERT Classifier)
# =========================
# ==========================================

model_name2 = "distilbert-base-uncased"

tokenizer2 = AutoTokenizer.from_pretrained(model_name2)

# NOTE: In real project, load YOUR trained model checkpoint
model2 = AutoModelForSequenceClassification.from_pretrained(
    model_name2,
    num_labels=2
)

def predict(text):
    inputs = tokenizer2(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        out = model2(**inputs)
    return torch.argmax(out.logits).item()

print("\n================ AFTER FINE-TUNING (Classifier) ================\n")

for i, r in enumerate(resumes):
    pred = predict(r)
    label = "MATCH" if pred == 1 else "NOT MATCH"
    print(f"Resume {i+1}: {label}")

# ==========================================
# SUMMARY METRICS (Simple Evaluation)
# ==========================================

print("\n================ COMPARISON SUMMARY ================\n")

print("MODEL 1 (MiniLM Similarity):")
print("- Output: Continuous score (0 to 1)")
print("- Metric: Cosine similarity")
print("- Strength: No training needed")
print("- Weakness: Not task-specific")

print("\nMODEL 2 (Fine-tuned DistilBERT):")
print("- Output: Classification (0/1)")
print("- Metric: Accuracy, Precision, Recall")
print("- Strength: Task-specific learning")
print("- Weakness: Needs training data")

# ==========================================
# SIMPLE ACCURACY DEMO (toy evaluation)
# ==========================================

y_true = [0, 1, 0]  # assumed correct labels

y_pred = [predict(r) for r in resumes]

accuracy = np.mean(np.array(y_pred) == np.array(y_true))

print("\nToy Accuracy (Fine-tuned model):", accuracy)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



================ BEFORE FINE-TUNING (MiniLM) ================

Resume 1 Score: 0.4088
Resume 2 Score: 0.7791
Resume 3 Score: 0.3598


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



================ AFTER FINE-TUNING (Classifier) ================

Resume 1: NOT MATCH
Resume 2: NOT MATCH
Resume 3: NOT MATCH

================ COMPARISON SUMMARY ================

MODEL 1 (MiniLM Similarity):
- Output: Continuous score (0 to 1)
- Metric: Cosine similarity
- Strength: No training needed
- Weakness: Not task-specific

MODEL 2 (Fine-tuned DistilBERT):
- Output: Classification (0/1)
- Metric: Accuracy, Precision, Recall
- Strength: Task-specific learning
- Weakness: Needs training data

Toy Accuracy (Fine-tuned model): 0.6666666666666666
